In [1]:
import duckdb
import pandas as pd

# Connect to DuckDB (creates a local database file)
con = duckdb.connect("E:/nyc-fhvhv-pipeline/nyc_trips.db")

# Load parquet directly into DuckDB
con.execute("""
    CREATE OR REPLACE TABLE trips AS 
    SELECT * FROM read_parquet('E:/nyc-fhvhv-pipeline/fhvhv_tripdata_2026-01.parquet')
    WHERE trip_miles > 0 
    AND trip_time > 0
    AND driver_pay > 0
""")

# Verify it loaded
result = con.execute("SELECT COUNT(*) as total_trips FROM trips").fetchdf()
print("✅ Trips loaded into DuckDB:")
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Trips loaded into DuckDB:
   total_trips
0     20885417


In [2]:
# Query 1 - Uber vs Lyft market share
print("=== UBER VS LYFT MARKET SHARE ===")
q1 = con.execute("""
    SELECT 
        CASE WHEN hvfhs_license_num = 'HV0003' THEN 'Uber' ELSE 'Lyft' END as carrier,
        COUNT(*) as total_trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as market_share_pct,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay,
        ROUND(AVG(trip_miles), 2) as avg_miles
    FROM trips
    GROUP BY hvfhs_license_num
    ORDER BY total_trips DESC
""").fetchdf()
print(q1)

# Query 2 - Busiest hours
print("\n=== TOP 5 BUSIEST HOURS ===")
q2 = con.execute("""
    SELECT 
        HOUR(pickup_datetime) as hour,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay
    FROM trips
    GROUP BY hour
    ORDER BY total_trips DESC
    LIMIT 5
""").fetchdf()
print(q2)

=== UBER VS LYFT MARKET SHARE ===
  carrier  total_trips  market_share_pct  avg_driver_pay  avg_miles
0    Uber     15192402             72.74           20.18       4.70
1    Lyft      5693015             27.26           19.09       4.48

=== TOP 5 BUSIEST HOURS ===
   hour  total_trips  avg_driver_pay
0    18      1267577           19.22
1    19      1227877           18.67
2    17      1218248           20.18
3    20      1141299           19.31
4    16      1097945           20.77


In [3]:
# Query 3 - Best paying pickup zones (Top 10)
print("=== TOP 10 BEST PAYING ZONES ===")
q3 = con.execute("""
    SELECT 
        PULocationID as zone_id,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay,
        ROUND(AVG(trip_miles), 2) as avg_miles
    FROM trips
    GROUP BY PULocationID
    ORDER BY avg_driver_pay DESC
    LIMIT 10
""").fetchdf()
print(q3)

# Query 4 - Day of week analysis
print("\n=== TRIPS BY DAY OF WEEK ===")
q4 = con.execute("""
    SELECT 
        DAYNAME(pickup_datetime) as day_of_week,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay
    FROM trips
    GROUP BY day_of_week
    ORDER BY total_trips DESC
""").fetchdf()
print(q4)

=== TOP 10 BEST PAYING ZONES ===
   zone_id  total_trips  avg_driver_pay  avg_miles
0      132       330178           56.07      17.89
1      199            4           45.99      15.38
2      138       364599           40.43      11.34
3      194         2298           39.24       6.89
4      110           31           35.21      10.89
5       27          770           29.76       9.38
6       46         3255           27.92       8.80
7      230       231068           27.35       6.07
8      261        66325           27.21       6.47
9       88        49225           26.87       6.64

=== TRIPS BY DAY OF WEEK ===
  day_of_week  total_trips  avg_driver_pay
0    Saturday      4045070           18.59
1      Friday      3753339           19.90
2    Thursday      3508305           21.34
3   Wednesday      2606369           20.58
4     Tuesday      2536337           20.62
5      Sunday      2237248           18.77
6      Monday      2198749           19.39


In [4]:
print("=== TRIP DISTANCE BREAKDOWN ===")
q5 = con.execute("""
    SELECT 
        CASE 
            WHEN trip_miles < 2 THEN '0-2 miles (Short)'
            WHEN trip_miles < 5 THEN '2-5 miles (Medium)'
            WHEN trip_miles < 10 THEN '5-10 miles (Long)'
            ELSE '10+ miles (Very Long)'
        END as trip_category,
        COUNT(*) as total_trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay
    FROM trips
    GROUP BY trip_category
    ORDER BY total_trips DESC
""").fetchdf()
print(q5)

=== TRIP DISTANCE BREAKDOWN ===
           trip_category  total_trips    pct  avg_driver_pay
0      0-2 miles (Short)      8076133  38.67            8.89
1     2-5 miles (Medium)      6747327  32.31           17.17
2      5-10 miles (Long)      3662510  17.54           27.97
3  10+ miles (Very Long)      2399447  11.49           52.17


In [5]:
print("=== DRIVER EARNINGS BREAKDOWN ===")
q6 = con.execute("""
    SELECT
        CASE
            WHEN driver_pay < 10 THEN 'Under $10'
            WHEN driver_pay < 20 THEN '$10-$20'
            WHEN driver_pay < 30 THEN '$20-$30'
            WHEN driver_pay < 50 THEN '$30-$50'
            ELSE 'Over $50'
        END as pay_bracket,
        COUNT(*) as total_trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM trips
    GROUP BY pay_bracket
    ORDER BY total_trips DESC
""").fetchdf()
print(q6)

=== DRIVER EARNINGS BREAKDOWN ===
  pay_bracket  total_trips    pct
0     $10-$20      7210442  34.52
1   Under $10      6232577  29.84
2     $20-$30      3669583  17.57
3     $30-$50      2661614  12.74
4    Over $50      1111201   5.32


In [6]:
# Zone 132 = JFK, Zone 138 = LaGuardia, Zone 1 = Newark
print("=== AIRPORT VS REGULAR TRIPS ===")
q7 = con.execute("""
    SELECT
        CASE 
            WHEN PULocationID IN (132, 138, 1) THEN 'Airport Pickup'
            WHEN DOLocationID IN (132, 138, 1) THEN 'Airport Dropoff'
            ELSE 'Regular Trip'
        END as trip_type,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay,
        ROUND(AVG(trip_miles), 2) as avg_miles
    FROM trips
    GROUP BY trip_type
    ORDER BY avg_driver_pay DESC
""").fetchdf()
print(q7)

=== AIRPORT VS REGULAR TRIPS ===
         trip_type  total_trips  avg_driver_pay  avg_miles
0   Airport Pickup       694779           47.86      14.45
1  Airport Dropoff       835183           42.23      12.52
2     Regular Trip     19355455           17.92       3.95


In [7]:
print("=== BEST HOURS TO DRIVE (BY EARNINGS) ===")
q8 = con.execute("""
    SELECT 
        HOUR(pickup_datetime) as hour,
        ROUND(AVG(driver_pay), 2) as avg_pay,
        ROUND(AVG(trip_miles), 2) as avg_miles,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay) / (AVG(trip_time)/3600), 2) as earnings_per_hour
    FROM trips
    GROUP BY hour
    ORDER BY earnings_per_hour DESC
""").fetchdf()
print(q8)

=== BEST HOURS TO DRIVE (BY EARNINGS) ===
    hour  avg_pay  avg_miles  total_trips  earnings_per_hour
0      4    21.34       6.45       321164              73.28
1      2    19.94       5.02       386509              72.99
2      5    22.25       6.68       377888              72.31
3      1    19.49       4.86       515653              71.21
4      3    20.00       5.68       312953              71.07
5      6    22.05       5.88       582857              69.43
6      0    19.03       4.94       729862              67.78
7     22    19.67       4.96      1045076              66.86
8     21    19.71       4.91      1075793              66.84
9     23    19.11       4.94       937344              66.51
10     7    21.29       4.74       886173              66.11
11    20    19.31       4.72      1141299              64.92
12     8    20.19       4.21      1087210              64.12
13    19    18.67       4.25      1227877              62.46
14     9    19.13       4.32       990398  

In [8]:
print("=== TIPS ANALYSIS ===")
q9 = con.execute("""
    SELECT
        CASE WHEN hvfhs_license_num = 'HV0003' THEN 'Uber' ELSE 'Lyft' END as carrier,
        COUNT(*) as total_trips,
        SUM(CASE WHEN tips > 0 THEN 1 ELSE 0 END) as trips_with_tips,
        ROUND(SUM(CASE WHEN tips > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tip_rate_pct,
        ROUND(AVG(CASE WHEN tips > 0 THEN tips END), 2) as avg_tip_when_tipped
    FROM trips
    GROUP BY carrier
    ORDER BY tip_rate_pct DESC
""").fetchdf()
print(q9)

=== TIPS ANALYSIS ===
  carrier  total_trips  trips_with_tips  tip_rate_pct  avg_tip_when_tipped
0    Lyft      5693015        1210931.0         21.27                 5.98
1    Uber     15192402        2854674.0         18.79                 5.99


In [9]:
print("=== WEEKLY TRIP TRENDS ===")
q10 = con.execute("""
    SELECT 
        WEEK(pickup_datetime) as week_number,
        COUNT(*) as total_trips,
        ROUND(AVG(driver_pay), 2) as avg_driver_pay,
        ROUND(SUM(driver_pay), 2) as total_driver_earnings
    FROM trips
    GROUP BY week_number
    ORDER BY week_number
""").fetchdf()
print(q10)

=== WEEKLY TRIP TRENDS ===
   week_number  total_trips  avg_driver_pay  total_driver_earnings
0            1      2543158           19.59            49818187.91
1            2      4521188           19.15            86596565.04
2            3      4841797           19.35            93711259.50
3            4      4610320           19.15            88308496.68
4            5      4368954           22.17            96843827.76


In [10]:
import os
os.makedirs("E:/nyc-fhvhv-pipeline/data/outputs", exist_ok=True)

q1.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/market_share.csv", index=False)
q2.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/busiest_hours.csv", index=False)
q3.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/best_paying_zones.csv", index=False)
q4.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/trips_by_day.csv", index=False)
q5.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/trip_distance_breakdown.csv", index=False)
q6.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/driver_earnings_brackets.csv", index=False)
q7.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/airport_vs_regular.csv", index=False)
q8.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/best_hours_to_drive.csv", index=False)
q9.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/tips_analysis.csv", index=False)
q10.to_csv("E:/nyc-fhvhv-pipeline/data/outputs/weekly_trends.csv", index=False)

print("✅ All 10 analyses saved!")
for f in os.listdir("E:/nyc-fhvhv-pipeline/data/outputs"):
    print(f"  - {f}")

✅ All 10 analyses saved!
  - airport_vs_regular.csv
  - best_hours_to_drive.csv
  - best_paying_zones.csv
  - busiest_hours.csv
  - driver_earnings_brackets.csv
  - market_share.csv
  - tips_analysis.csv
  - trips_by_day.csv
  - trip_distance_breakdown.csv
  - weekly_trends.csv
